<a href="https://colab.research.google.com/github/brittanybee-alt/Updated-Capstone-Project---Brittany-Barrion/blob/main/BB_DC_Capstone_Two_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Two: Preprocessing and Training Data Development
## Where You Grow Up Matters: Predicting Adult Income for Low-Income Children in the DC Region

**Project question:** Which neighborhood characteristics best predict adult income for children raised in low-income families in the DC metro area?

### Data Source
All data comes from **The Opportunity Atlas** (Opportunity Insights and the U.S. Census Bureau):

**[The Opportunity Atlas: household income at age 35 for children raised in low-income families, DC region](https://www.opportunityatlas.org/?%7B%22mapping%22%3A%22kfr%22%2C%22dataToggles%22%3A%7B%22parentIncome%22%3A%22p25%22%2C%22childRace%22%3A%22rP%22%2C%22childGender%22%3A%22gP%22%2C%22cohortTimeline%22%3A%22e%22%7D%2C%22compareDataToggles%22%3A%7B%22parentIncome%22%3A%22pall%22%2C%22childRace%22%3A%22rP%22%2C%22childGender%22%3A%22gP%22%2C%22cohortTimeline%22%3A%22e%22%7D%2C%22selectedChapterId%22%3A%22chapter1%22%2C%22compareFeatures%22%3A%22loc%22%2C%22selectedFeatureByID%22%3A11001010200%2C%22filters%22%3A%5B%5D%2C%22mode%22%3A%22standardMode%22%2C%22mapBounds%22%3A%5B%5B-77.29824387656735%2C38.72686964592006%5D%2C%5B-76.79125493105334%2C39.0687791383437%5D%5D%2C%22floatingPanelIsMinimized%22%3Afalse%2C%22showStoryMarkers%22%3Atrue%2C%22showHistogram%22%3Afalse%2C%22propertyShownOnMap%22%3A%22kfr_rP_gP_p25_e%22%7D)**

Two Census-tract-level files were downloaded from this map view and joined in the Data Wrangling notebook:
- `shown_tract_kfr_rP_gP_p25.csv`: average household income at age 35 for children whose parents were at the 25th income percentile (the **target**)
- `shown_tract_allCharacteristics.csv`: neighborhood characteristics by tract (the **predictors**)

### Goal
Create a cleaned development dataset to use in the modeling step.

**Input:** `data/processed/eda_features.csv` (752 tracts × 14 columns), the output of the EDA notebook. It contains the target, the 9 numeric features selected in EDA (with population density already log-transformed) and `state`.

**Steps (following the outline):**
1. Create dummy or indicator features for categorical variables
2. Standardize the magnitude of numeric features using a scaler
3. Split into testing and training datasets

### Setup
Import libraries and load the cleaned dataset.

**In Google Colab:** upload `eda_features.csv` into a `data/processed` folder (Files panel on the left), then run the cells below. The next cell creates the folder if it doesn't exist yet.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 50)
os.makedirs('data/processed', exist_ok=True)

df = pd.read_csv('/content/eda_features.csv', dtype={'tract': str})
print(df.shape)
df.head()

(752, 14)


,tract,name,state,county,income_age35,median_rent,median_hh_income_1990,poverty_rate,frac_nonwhite,frac_foreign_born,frac_single_parents,log_pop_density,frac_short_commute,census_response_rate
0,51059480203,"Tysons, VA",VA,"Fairfax County, VA",69199.0,2138.0,128382,0.0673,0.3958,0.6029,0.1672,9.565564,0.1916,76.2
1,24031701002,"Rockville, MD",MD,"Montgomery County, MD",69034.0,3193.0,169391,0.0262,0.2914,0.2444,0.0787,8.176110,0.1747,88.3
2,51013100400,"Arlington, VA",VA,"Arlington County, VA",68322.0,3501.0,162342,0.0286,0.1214,0.1138,0.0941,7.765993,0.2106,91.2
3,51013101601,"North Highland, Arlington, VA",VA,"Arlington County, VA",65785.0,1982.0,74114,0.1455,0.2323,0.1876,0.2963,8.496990,0.1230,79.5
4,51510200702,"Eisenhower East, Alexandria, VA",VA,"Alexandria city, VA",64583.0,2186.0,64247,0.0222,0.3072,0.1373,0.4101,9.041448,0.1798,82.1


In [2]:
df.dtypes

,0
tract,object
name,object
state,object
county,object
income_age35,float64
median_rent,float64
median_hh_income_1990,int64
poverty_rate,float64
frac_nonwhite,float64
frac_foreign_born,float64


### Choosing the target and features
- **Target:** `income_age35`, the average household income at age 35 for children from low-income families.
- **ID and label columns not used as features:** `tract` is an ID number, and `name` is a place label shared by many tracts (only 189 unique names for 752 tracts). Neither describes a neighborhood characteristic, so both are set aside.
- **Features:** the 9 numeric features kept after EDA's feature selection, plus `state`.
- **`county` is set aside in favor of `state`.** Several counties have only 2 or 3 tracts (Anne Arundel, Fairfax city, Falls Church city), too few for a model to learn a reliable county effect. `state` captures the main regional differences with plenty of tracts in each group.

In [3]:
TARGET = 'income_age35'
ID_COLS = ['tract', 'name', 'county']
CAT_COLS = ['state']

print(df['state'].value_counts())

state
MD    304
VA    273
DC    175
Name: count, dtype: int64


## Step 1: Create Dummy or Indicator Features for Categorical Variables

**Review question: Does my dataset have any categorical data, such as gender or day of the week?**

Yes. **`state`** (DC, MD or VA) is categorical: it names a place and has no numeric order. A model can't use text, and coding the states as 1, 2, 3 would wrongly suggest that VA is "more" than MD. Instead, I use `pd.get_dummies()` to create 0/1 indicator columns.

**DC is the baseline category.** With `drop_first=True`, the DC column is dropped, leaving `state_MD` and `state_VA`. A tract in DC has 0 in both columns. This avoids redundant columns (the "dummy variable trap"), which matters for linear regression, and each coefficient can be read as "compared with DC," the focus of the project.

In [4]:
# Put DC first so it's the category that gets dropped
df['state'] = pd.Categorical(df['state'], categories=['DC', 'MD', 'VA'])

df = pd.get_dummies(df, columns=CAT_COLS, drop_first=True, dtype=int)

dummy_cols = ['state_MD', 'state_VA']
print('Dummy columns created:', dummy_cols)
print()
print('Tracts per group:')
print('  DC (baseline):', ((df['state_MD'] == 0) & (df['state_VA'] == 0)).sum())
print('  MD:           ', df['state_MD'].sum())
print('  VA:           ', df['state_VA'].sum())

Dummy columns created: ['state_MD', 'state_VA']

Tracts per group:
  DC (baseline): 175
  MD:            304
  VA:            273


## Step 2: Standardize the Magnitude of Numeric Features Using a Scaler

**Review question: Do my features have values that range from 0-100, 0-1, or both and more?**

Both and more. The table below shows the ranges:
- **0 to 1:** the share features (`poverty_rate`, `frac_nonwhite`, `frac_foreign_born`, `frac_single_parents`, `frac_short_commute`)
- **0 to 100:** `census_response_rate`
- **Around 4 to 11:** `log_pop_density`, which was log-transformed in EDA
- **Much larger:** dollar amounts (`median_rent` from \$241 to \$3,501, and `median_hh_income_1990` from about \$11,000 to \$272,000)

In [5]:
num_cols = [c for c in df.columns if c not in [TARGET] + ID_COLS + dummy_cols]
print(len(num_cols), 'numeric features:', num_cols)

df[num_cols].describe().T[['min', 'max', 'mean', '50%', 'std']].round(3)

9 numeric features: ['median_rent', 'median_hh_income_1990', 'poverty_rate', 'frac_nonwhite', 'frac_foreign_born', 'frac_single_parents', 'log_pop_density', 'frac_short_commute', 'census_response_rate']


,min,max,mean,50%,std
median_rent,241.000,3501.000,1750.248,1693.500,594.978
median_hh_income_1990,11176.000,272105.000,87444.848,82629.500,37105.614
poverty_rate,0.002,0.695,0.105,0.078,0.096
frac_nonwhite,0.068,0.998,0.604,0.598,0.290
frac_foreign_born,0.000,0.711,0.231,0.205,0.150
frac_single_parents,0.000,1.000,0.329,0.275,0.238
log_pop_density,4.463,11.111,8.732,8.767,0.954
frac_short_commute,0.000,0.704,0.126,0.118,0.069
census_response_rate,49.000,92.600,78.959,79.400,7.023


**Why this matters:** without standardizing, a feature measured in dollars (such as 1990 household income, with a standard deviation near \$37,000) would carry far more weight than a share measured from 0 to 1. Penalty-based models such as ridge and lasso would then treat features unfairly, and coefficients couldn't be compared to see which feature matters most.

**Which columns get scaled:** I use `StandardScaler` to rescale each of the 9 numeric features to a mean of 0 and a standard deviation of 1. I do **not** scale:
- **the target**, so predictions and errors stay in real dollars and are easy to interpret
- **the dummy columns**, since 0/1 values are already on a small, meaningful scale

**The scaler is created here but fit only on the training set in Step 3.** If it were fit on all the data, the test set's averages would leak into training and make the model look better than it really is.

In [6]:
scaler = preprocessing.StandardScaler()

## Step 3: Split into Testing and Training Datasets

I split the data **80/20** with `train_test_split`, using `random_state=42` so the split is the same every time the notebook runs. The models will learn from the **training set**, and the **test set** is held back until the end to judge how well they predict tracts they haven't seen.

I also **stratify by state**, so DC, Maryland and Virginia tracts appear in the same proportions in both sets. Otherwise, a random split could put too few DC tracts in the test set to judge how well the model works for DC.

In [7]:
X = df.drop(columns=[TARGET] + ID_COLS)
y = df[TARGET]

state_group = np.select([df['state_MD'] == 1, df['state_VA'] == 1], ['MD', 'VA'], default='DC')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=state_group)

print('Training set:', X_train.shape)
print('Testing set: ', X_test.shape)

Training set: (601, 11)
Testing set:  (151, 11)


In [8]:
# Check the state mix is the same in both sets
def state_mix(X_part):
    g = np.select([X_part['state_MD'] == 1, X_part['state_VA'] == 1], ['MD', 'VA'], default='DC')
    return pd.Series(g).value_counts(normalize=True).round(3)

pd.DataFrame({'train': state_mix(X_train), 'test': state_mix(X_test)})

,train,test
MD,0.404,0.404
VA,0.363,0.364
DC,0.233,0.232


Now the scaler is **fit on the training data only** and then used to transform both sets.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

**Check:** the training features should now have a mean of about 0 and a standard deviation of about 1. The test features will be close to, but not exactly, 0 and 1, because they were scaled using the training set's averages. That's expected and correct.

In [10]:
pd.DataFrame({
    'train_mean': X_train[num_cols].mean(),
    'train_std': X_train[num_cols].std(),
    'test_mean': X_test[num_cols].mean(),
    'test_std': X_test[num_cols].std(),
}).round(2)

,train_mean,train_std,test_mean,test_std
median_rent,0.0,1.0,0.01,0.98
median_hh_income_1990,0.0,1.0,-0.07,0.95
poverty_rate,0.0,1.0,0.04,1.01
frac_nonwhite,0.0,1.0,0.10,1.05
frac_foreign_born,-0.0,1.0,-0.16,0.92
frac_single_parents,-0.0,1.0,0.22,1.09
log_pop_density,0.0,1.0,0.07,0.90
frac_short_commute,-0.0,1.0,0.01,0.98
census_response_rate,0.0,1.0,0.02,0.92


### Save the preprocessed data
These files are the inputs for the Modeling notebook, so every model uses exactly the same split and scaling.

In [11]:
X_train.to_csv('data/processed/X_train.csv', index=False)
X_test.to_csv('data/processed/X_test.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)
print('Saved X_train, X_test, y_train and y_test to data/processed/')
print('Final feature columns:', X_train.columns.tolist())

Saved X_train, X_test, y_train and y_test to data/processed/
Final feature columns: ['median_rent', 'median_hh_income_1990', 'poverty_rate', 'frac_nonwhite', 'frac_foreign_born', 'frac_single_parents', 'log_pop_density', 'frac_short_commute', 'census_response_rate', 'state_MD', 'state_VA']


## Summary
- **Input:** `eda_features.csv` from EDA (752 tracts, 9 selected numeric features plus `state`). The target is `income_age35`. `tract`, `name` and `county` were set aside as ID or label columns.
- **Step 1, dummies:** `state` was converted to `state_MD` and `state_VA`, with **DC as the baseline**.
- **Step 2, scaling:**
  - The features range from 0-1 (shares) to 0-100 (response rate) to hundreds of thousands (dollars), so scaling was needed.
  - The 9 numeric features were standardized with `StandardScaler`, fit on the training data only. The target and dummy columns were not scaled.
- **Step 3, split:** 80/20 train/test split, stratified by state, with `random_state=42`. The result is **601 training tracts and 151 testing tracts**, each with **11 features** (9 numeric + 2 state dummies).
- **Output:** `X_train`, `X_test`, `y_train` and `y_test` saved to `data/processed/` for the Modeling step.